# OOF recovery utilities

Utilities for combining completed fold files, recomputing metrics, and adding student identifiers to legacy AKT OOF outputs.


In [ ]:
# AKT First Attempt Helper 
# Creates "all" file when a fold is missing and rerun afterwards and then prints final metrics

# -------------------------
# Outer CV using fixed folds
# -------------------------
# Restart only outer fold 3 - Change in AKT code
# folds_to_run = [3]
# 
# for fold in folds_to_run:
    
    
import os
import numpy as np
import pandas as pd

from statistics import mean, stdev
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# SETTINGS — CHANGE KC NAME WHEN NEEDED
# ============================================================

model_name = "AKT"
kc_name = "itemid"          # Example: itemid, propertyid, exerciseid
n_splits = 3
oof_output_dir = "AE_FA"


# ============================================================
# HELPER
# ============================================================

def rmse_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sqrt(
            np.mean((y_true - y_pred) ** 2)
        )
    )


# ============================================================
# LOAD ALL SAVED FOLD FILES
# ============================================================

all_fold_dfs = []

all_labels_all_folds = []
all_preds_all_folds = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []


for fold in range(1, n_splits + 1):

    fold_file = os.path.join(
        oof_output_dir,
        f"oof_{model_name}_{kc_name}_fold{fold}.csv"
    )

    if not os.path.exists(fold_file):
        raise FileNotFoundError(
            f"Missing Fold {fold} file:\n{fold_file}"
        )

    fold_df = pd.read_csv(fold_file)

    required_columns = {
        "row_id",
        "fold",
        "y_true",
        "y_pred"
    }

    missing_columns = required_columns - set(fold_df.columns)

    if missing_columns:
        raise ValueError(
            f"Fold {fold} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    fold_labels = fold_df["y_true"].astype(int).to_numpy()
    fold_preds = fold_df["y_pred"].astype(float).to_numpy()
    fold_binary_preds = (fold_preds > 0.5).astype(int)

    fold_auc = (
        roc_auc_score(fold_labels, fold_preds)
        if len(np.unique(fold_labels)) > 1
        else np.nan
    )

    fold_acc = accuracy_score(
        fold_labels,
        fold_binary_preds
    )

    fold_rmse = rmse_score(
        fold_labels,
        fold_preds
    )

    fold_mae = mean_absolute_error(
        fold_labels,
        fold_preds
    )

    fold_precision = precision_score(
        fold_labels,
        fold_binary_preds,
        zero_division=0
    )

    fold_recall = recall_score(
        fold_labels,
        fold_binary_preds,
        zero_division=0
    )

    fold_f1 = f1_score(
        fold_labels,
        fold_binary_preds,
        zero_division=0
    )

    auc_per_fold.append(fold_auc)
    acc_per_fold.append(fold_acc)
    rmse_per_fold.append(fold_rmse)
    mae_per_fold.append(fold_mae)
    precision_per_fold.append(fold_precision)
    recall_per_fold.append(fold_recall)
    f1_per_fold.append(fold_f1)

    all_labels_all_folds.extend(fold_labels.tolist())
    all_preds_all_folds.extend(fold_preds.tolist())
    all_fold_dfs.append(fold_df)

    print(f"\nEvaluation on Fold {fold} Test Set:")
    print(f"AUC: {fold_auc:.6f}")
    print(f"Accuracy: {fold_acc:.6f}")
    print(f"RMSE: {fold_rmse:.6f}")
    print(f"MAE: {fold_mae:.6f}")
    print(f"Precision: {fold_precision:.6f}")
    print(f"Recall: {fold_recall:.6f}")
    print(f"F1 Score: {fold_f1:.6f}")


# ============================================================
# REBUILD AND SAVE COMBINED OOF FILE
# ============================================================

oof_df = (
    pd.concat(
        all_fold_dfs,
        ignore_index=True
    )
    .sort_values("row_id")
    .reset_index(drop=True)
)

if oof_df["row_id"].duplicated().any():
    duplicate_count = int(
        oof_df["row_id"].duplicated().sum()
    )

    raise ValueError(
        f"The combined OOF data contains "
        f"{duplicate_count} duplicate row_id values."
    )

oof_outfile = os.path.join(
    oof_output_dir,
    f"oof_{model_name}_{kc_name}_all.csv"
)

oof_df.to_csv(
    oof_outfile,
    index=False
)

print(
    f"\nSaved combined OOF predictions to "
    f"{oof_outfile}"
)


# ============================================================
# POOLED AND FOLD-AVERAGED RESULTS
# ============================================================

if len(all_labels_all_folds) > 0:

    binary_preds = [
        1 if p > 0.5 else 0
        for p in all_preds_all_folds
    ]

    pooled_auc = roc_auc_score(
        all_labels_all_folds,
        all_preds_all_folds
    )

    pooled_acc = accuracy_score(
        all_labels_all_folds,
        binary_preds
    )

    pooled_rmse = rmse_score(
        all_labels_all_folds,
        all_preds_all_folds
    )

    pooled_mae = mean_absolute_error(
        all_labels_all_folds,
        all_preds_all_folds
    )

    pooled_precision = precision_score(
        all_labels_all_folds,
        binary_preds,
        zero_division=0
    )

    pooled_recall = recall_score(
        all_labels_all_folds,
        binary_preds,
        zero_division=0
    )

    pooled_f1 = f1_score(
        all_labels_all_folds,
        binary_preds,
        zero_division=0
    )

    def mean_std(values):
        return (
            mean(values),
            stdev(values) if len(values) > 1 else 0.0
        )

    avg_auc, std_auc = mean_std(auc_per_fold)
    avg_acc, std_acc = mean_std(acc_per_fold)
    avg_rmse, std_rmse = mean_std(rmse_per_fold)
    avg_mae, std_mae = mean_std(mae_per_fold)
    avg_precision, std_precision = mean_std(
        precision_per_fold
    )
    avg_recall, std_recall = mean_std(
        recall_per_fold
    )
    avg_f1, std_f1 = mean_std(f1_per_fold)

    print("\n=== Final Evaluation Across All Folds ===")
    print("Metric       | Pooled Score | Average Score ± Std")
    print("-------------|--------------|---------------------")
    print(
        f"AUC          | {pooled_auc:.6f}      | "
        f"{avg_auc:.6f} ± {std_auc:.6f}"
    )
    print(
        f"Accuracy     | {pooled_acc:.6f}      | "
        f"{avg_acc:.6f} ± {std_acc:.6f}"
    )
    print(
        f"RMSE         | {pooled_rmse:.6f}      | "
        f"{avg_rmse:.6f} ± {std_rmse:.6f}"
    )
    print(
        f"MAE          | {pooled_mae:.6f}      | "
        f"{avg_mae:.6f} ± {std_mae:.6f}"
    )
    print(
        f"Precision    | {pooled_precision:.6f}      | "
        f"{avg_precision:.6f} ± {std_precision:.6f}"
    )
    print(
        f"Recall       | {pooled_recall:.6f}      | "
        f"{avg_recall:.6f} ± {std_recall:.6f}"
    )
    print(
        f"F1 Score     | {pooled_f1:.6f}      | "
        f"{avg_f1:.6f} ± {std_f1:.6f}"
    )

else:
    print("No valid predictions across all folds.")

In [ ]:
# SAKT First Attempt Helper
# Prints final metrics for SAKT by loading the oof files

import os
from statistics import mean, stdev

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    f1_score
)


# --------------------------------------------------
# Settings
# --------------------------------------------------
oof_folder = "SAKT_FA"
model_name = "SAKT"
kc_name = "exerciseid"
n_splits = 3


# --------------------------------------------------
# Helper functions
# --------------------------------------------------
def rmse_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sqrt(
            np.mean((y_true - y_pred) ** 2)
        )
    )


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=float)

    binary_preds = (y_pred > 0.5).astype(int)

    auc = (
        roc_auc_score(y_true, y_pred)
        if len(np.unique(y_true)) > 1
        else np.nan
    )

    return {
        "auc": auc,
        "accuracy": accuracy_score(
            y_true,
            binary_preds
        ),
        "rmse": rmse_score(
            y_true,
            y_pred
        ),
        "mae": mean_absolute_error(
            y_true,
            y_pred
        ),
        "precision": precision_score(
            y_true,
            binary_preds,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            binary_preds,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            binary_preds,
            zero_division=0
        )
    }


def mean_std(values):
    valid_values = [
        float(value)
        for value in values
        if not pd.isna(value)
    ]

    if not valid_values:
        return np.nan, np.nan

    return (
        mean(valid_values),
        stdev(valid_values)
        if len(valid_values) > 1
        else 0.0
    )


# --------------------------------------------------
# Load the three fold files
# --------------------------------------------------
all_fold_dfs = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []


for fold in range(1, n_splits + 1):

    fold_file = os.path.join(
        oof_folder,
        f"oof_{model_name}_{kc_name}_fold{fold}.csv"
    )

    if not os.path.exists(fold_file):
        raise FileNotFoundError(
            f"Missing fold file: {fold_file}"
        )

    fold_df = pd.read_csv(fold_file)

    required_columns = {
        "row_id",
        "y_true",
        "y_pred"
    }

    missing_columns = (
        required_columns - set(fold_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{fold_file} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    if fold_df["row_id"].duplicated().any():
        raise ValueError(
            f"{fold_file} contains duplicate row_id values."
        )

    fold_metrics = calculate_metrics(
        fold_df["y_true"],
        fold_df["y_pred"]
    )

    auc_per_fold.append(fold_metrics["auc"])
    acc_per_fold.append(fold_metrics["accuracy"])
    rmse_per_fold.append(fold_metrics["rmse"])
    mae_per_fold.append(fold_metrics["mae"])
    precision_per_fold.append(
        fold_metrics["precision"]
    )
    recall_per_fold.append(
        fold_metrics["recall"]
    )
    f1_per_fold.append(
        fold_metrics["f1"]
    )

    all_fold_dfs.append(fold_df)

    print(f"\nEvaluation on Fold {fold} Test Set:")
    print(f"AUC: {fold_metrics['auc']:.6f}")
    print(f"Accuracy: {fold_metrics['accuracy']:.6f}")
    print(f"RMSE: {fold_metrics['rmse']:.6f}")
    print(f"MAE: {fold_metrics['mae']:.6f}")
    print(f"Precision: {fold_metrics['precision']:.6f}")
    print(f"Recall: {fold_metrics['recall']:.6f}")
    print(f"F1 Score: {fold_metrics['f1']:.6f}")


# --------------------------------------------------
# Pool all three folds in memory
# Nothing is saved or overwritten
# --------------------------------------------------
pooled_df = pd.concat(
    all_fold_dfs,
    ignore_index=True
)

if pooled_df["row_id"].duplicated().any():
    duplicate_count = int(
        pooled_df["row_id"].duplicated().sum()
    )

    raise ValueError(
        f"The three fold files contain "
        f"{duplicate_count} duplicated row_id values."
    )

pooled_metrics = calculate_metrics(
    pooled_df["y_true"],
    pooled_df["y_pred"]
)


# --------------------------------------------------
# Calculate fold averages and standard deviations
# --------------------------------------------------
avg_auc, std_auc = mean_std(auc_per_fold)
avg_acc, std_acc = mean_std(acc_per_fold)
avg_rmse, std_rmse = mean_std(rmse_per_fold)
avg_mae, std_mae = mean_std(mae_per_fold)

avg_precision, std_precision = mean_std(
    precision_per_fold
)

avg_recall, std_recall = mean_std(
    recall_per_fold
)

avg_f1, std_f1 = mean_std(
    f1_per_fold
)


# --------------------------------------------------
# Final pooled and averaged results
# --------------------------------------------------
print("\n=== Final Evaluation Across All Folds ===")
print("Metric       | Pooled Score | Average Score ± Std")
print("-------------|--------------|---------------------")

print(
    f"AUC          | {pooled_metrics['auc']:.6f}      | "
    f"{avg_auc:.6f} ± {std_auc:.6f}"
)

print(
    f"Accuracy     | {pooled_metrics['accuracy']:.6f}      | "
    f"{avg_acc:.6f} ± {std_acc:.6f}"
)

print(
    f"RMSE         | {pooled_metrics['rmse']:.6f}      | "
    f"{avg_rmse:.6f} ± {std_rmse:.6f}"
)

print(
    f"MAE          | {pooled_metrics['mae']:.6f}      | "
    f"{avg_mae:.6f} ± {std_mae:.6f}"
)

print(
    f"Precision    | {pooled_metrics['precision']:.6f}      | "
    f"{avg_precision:.6f} ± {std_precision:.6f}"
)

print(
    f"Recall       | {pooled_metrics['recall']:.6f}      | "
    f"{avg_recall:.6f} ± {std_recall:.6f}"
)

print(
    f"F1 Score     | {pooled_metrics['f1']:.6f}      | "
    f"{avg_f1:.6f} ± {std_f1:.6f}"
)

print(f"\nTotal pooled predictions: {len(pooled_df)}")



In [ ]:
# AKT Fixing 
# Adds studentid to the oof files if they ran on the old AKT oof code

import os
import glob
import numpy as np
import pandas as pd

# --------------------------------------------------
# Load the EXACT original dataset used for AKT
# --------------------------------------------------
# logging_data = pd.read_csv("path/to/dataset.csv")

student_col = "Anon Student Id"
correct_col = "eventualcorrect"

# Reproduce the same row_id creation used by the AKT code.
logging_data = logging_data.copy().reset_index(drop=True)

if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data))

# One lookup row per original interaction.
row_lookup = (
    logging_data[
        ["row_id", student_col, correct_col]
    ]
    .drop_duplicates(subset=["row_id"])
    .rename(
        columns={
            student_col: "student_id",
            correct_col: "original_y_true"
        }
    )
)

if not row_lookup["row_id"].is_unique:
    raise ValueError("row_id is not unique in the original dataset.")

# --------------------------------------------------
# Folder containing the old AKT OOF files
# --------------------------------------------------
oof_folder = "AE_AF"

oof_files = glob.glob(
    os.path.join(oof_folder, "oof_AKT_*.csv")
)

if not oof_files:
    raise FileNotFoundError(
        f"No AKT OOF CSV files found inside {oof_folder}"
    )

for oof_file in oof_files:
    # Avoid processing files already created by this script.
    if oof_file.endswith("_with_student_id.csv"):
        continue

    oof_df = pd.read_csv(oof_file)

    if "row_id" not in oof_df.columns:
        raise ValueError(
            f"{oof_file} does not contain row_id."
        )

    # Remove an existing student_id before rebuilding it.
    if "student_id" in oof_df.columns:
        oof_df = oof_df.drop(columns=["student_id"])

    enriched_df = oof_df.merge(
        row_lookup,
        on="row_id",
        how="left",
        validate="many_to_one"
    )

    # Every prediction must map to a student.
    missing_students = enriched_df["student_id"].isna().sum()

    if missing_students > 0:
        raise ValueError(
            f"{oof_file}: {missing_students} rows could not be "
            f"matched. The original dataset or its row order may differ."
        )

    # Strong safety check: verify that row_id points to the same label.
    label_mismatch = (
        enriched_df["y_true"].astype(int)
        != enriched_df["original_y_true"].astype(int)
    )

    if label_mismatch.any():
        raise ValueError(
            f"{oof_file}: {label_mismatch.sum()} y_true values "
            f"do not match the original dataset. Do not use this merge; "
            f"the row_id mapping is different."
        )

    enriched_df = enriched_df.drop(
        columns=["original_y_true"]
    )

    # Put student_id directly after row_id.
    ordered_columns = [
        "row_id",
        "student_id"
    ] + [
        column
        for column in enriched_df.columns
        if column not in ["row_id", "student_id"]
    ]

    enriched_df = enriched_df[ordered_columns]

    output_file = oof_file.replace(
        ".csv",
        "_with_student_id.csv"
    )

    enriched_df.to_csv(
        output_file,
        index=False
    )

    print(f"Saved: {output_file}")